# Train YOLO11n cho 10 Biển Báo Giao Thông (Xuất ONNX cho Xe Tự Hành)

**Danh sách 10 lớp (0 -> 9):**
- `0: left` (1. Chỉ được rẽ trái)
- `1: no_left` (2. Cấm rẽ trái)
- `2: right` (3. Chỉ được rẽ phải)
- `3: no_right` (4. Cấm rẽ phải)
- `4: straight` (5. Chỉ được đi thẳng)
- `5: stop` (6. Dừng lại STOP)
- `6: no_straight` (7. Cấm đi thẳng)
- `7: no_entry` (8. Cấm đi ngược chiều)
- `8: speed_30` (9. Tốc độ tối đa 30 km/h)
- `9: speed_60` (10. Tốc độ tối đa 60 km/h)

**Hướng dẫn nhanh trên Google Colab:**
1. Chọn **Runtime -> Change runtime type -> T4 GPU**.
2. Upload file `yolo11_10signs_dataset.zip` lên Google Drive (hoặc upload trực tiếp ở bước 2).
3. Chạy lần lượt các ô từ trên xuống. Sau khi xong, tải file `yolo11n_10signs.onnx` về và chép vào thư mục `P2.NhanDienBienBao/` của dự án.

In [ ]:
# 1. Kiểm tra GPU và cài đặt Ultralytics + ONNX
!nvidia-smi
!pip install -q ultralytics onnx onnxslim onnxruntime

In [ ]:
# 2. Nạp bộ dữ liệu yolo11_10signs_dataset.zip (Cách A: từ Google Drive | Cách B: Upload trực tiếp)
import os, zipfile

USE_GOOGLE_DRIVE = True  # Đổi thành False nếu muốn upload trực tiếp từ máy tính
DRIVE_ZIP_PATH = "/content/drive/MyDrive/yolo11_10signs_dataset.zip"
DATASET_DIR = "/content/yolo11_10signs_dataset"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = DRIVE_ZIP_PATH
else:
    from google.colab import files
    uploaded = files.upload()
    zip_path = next(iter(uploaded.keys()))

assert os.path.exists(zip_path), f"Khong tim thay file ZIP tai: {zip_path}"
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(DATASET_DIR)

# Ghi lại đường dẫn tuyệt đối trong data.yaml để Ultralytics đọc chắc chắn 100%
yaml_path = os.path.join(DATASET_DIR, "data.yaml")
with open(yaml_path, "r", encoding="utf-8") as f:
    yaml_text = f.read()
yaml_text = yaml_text.replace("path: .", f"path: {DATASET_DIR}")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_text)

print("Da giai nen xong! Noi dung data.yaml:")
print(yaml_text)

In [ ]:
# 3. Train YOLO11n (QUAN TRỌNG: fliplr=0.0, flipud=0.0 để không lật ngược biển Rẽ Trái <-> Rẽ Phải!)
from ultralytics import YOLO

IMG_SIZE = 416  # 416x416: rất nhanh trên CPU Laptop/Pi, vẫn giữ nét rõ chữ số 30/60 và mũi tên
model = YOLO("yolo11n.pt")

results = model.train(
    data=yaml_path,
    epochs=80,
    imgsz=IMG_SIZE,
    batch=32,
    patience=20,
    device=0,
    workers=2,
    project="/content/runs_signs",
    name="yolo11n_10signs",
    exist_ok=True,
    # Tắt lật ngang/dọc vì biển Rẽ Trái/Phải và số 30/60 không đối xứng!
    fliplr=0.0,
    flipud=0.0,
    # Tăng cường ánh sáng/màu/góc nghiêng nhẹ để chống tối, chói đèn và webcam nhạt màu
    hsv_h=0.015,
    hsv_s=0.60,
    hsv_v=0.50,
    degrees=10.0,
    scale=0.50,
    mosaic=0.80,
    close_mosaic=12,
)

In [ ]:
# 4. Đánh giá độ chính xác trên tập validation
best_pt = "/content/runs_signs/yolo11n_10signs/weights/best.pt"
best_model = YOLO(best_pt)
metrics = best_model.val(data=yaml_path, imgsz=IMG_SIZE)
print(f"mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# 5. Xuất sang ONNX (yolo11n_10signs.onnx) và tải về máy
import shutil
from google.colab import files

onnx_exported = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    opset=12,
    simplify=True,
    dynamic=False,
)

out_onnx = "/content/yolo11n_10signs.onnx"
shutil.copy2(onnx_exported, out_onnx)
print(f"Da xuat thanh cong: {out_onnx} ({os.path.getsize(out_onnx)/1024/1024:.2f} MB)")

# Sao lưu thêm lên Google Drive nếu đã mount Drive
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy2(out_onnx, "/content/drive/MyDrive/yolo11n_10signs.onnx")
    shutil.copy2(best_pt, "/content/drive/MyDrive/yolo11n_10signs_best.pt")
    print("Da sao luu len Google Drive: MyDrive/yolo11n_10signs.onnx")

files.download(out_onnx)